In [1]:
!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 15.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]


In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from scipy.interpolate import griddata
import warnings
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt

In [2]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [4]:
try:
    dt = xr.open_mfdataset("./data/ice_conc_nh*.nc",
                           combine = 'by_coords',
                           chunks ={'time':'auto', 'xc':'auto', 'yc':'auto'})
    print("Dataset loaded successfully.")
    print("\nDataset Information:")
    print(dt)
except FileNotFoundError:
    print("Dataset files not found. Please check the file path and ensure the files exist.")
    dt = None

Dataset loaded successfully.

Dataset Information:
<xarray.Dataset> Size: 231MB
Dimensions:                         (time: 28, nv: 2, yc: 432, xc: 432)
Coordinates:
  * time                            (time) datetime64[ns] 224B 2002-06-01T12:...
  * xc                              (xc) float64 3kB -5.388e+03 ... 5.388e+03
  * yc                              (yc) float64 3kB 5.388e+03 ... -5.388e+03
    lat                             (yc, xc) float32 746kB dask.array<chunksize=(432, 432), meta=np.ndarray>
    lon                             (yc, xc) float32 746kB dask.array<chunksize=(432, 432), meta=np.ndarray>
Dimensions without coordinates: nv
Data variables:
    Lambert_Azimuthal_Grid          (time) int32 112B -2147483647 ... -214748...
    time_bnds                       (time, nv) datetime64[ns] 448B dask.array<chunksize=(1, 2), meta=np.ndarray>
    ice_conc                        (time, yc, xc) float64 42MB dask.array<chunksize=(1, 432, 432), meta=np.ndarray>
    raw_ice_conc_v

In [7]:
print(f"Initial nans in 'ice_conc' variable: {dt['ice_conc'].isnull().sum().values}")

Initial nans in 'ice_conc' variable: 2487716


In [13]:
dt.isel(time=0)['ice_conc'].isnull().sum().values

array(88847)

In [ ]:
88847/186624

0.47607488854595337

In [18]:
#Temporal and Spatial Interpolation
ice_conc_rechunked = dt['ice_conc'].chunk({'time': -1, 'xc': 'auto', 'yc' : 'auto'})

ice_conc_temp_interp = ice_conc_rechunked.interpolate_na(dim='time', method='linear')
ice_conc_temp_interp = ice_conc_temp_interp.ffill(dim='time').bfill(dim='time')
dt['ice_conc_temp_interp'] = ice_conc_temp_interp

original_nans = dt['ice_conc'].isnull().sum().values

In [19]:
print(original_nans)

2487716


In [20]:
temp_interp_nans_count = dt['ice_conc_temp_interp'].isnull().sum().values
print(temp_interp_nans_count)

2487716


In [21]:
successfully_filled_nans_temp = dt['ice_conc'].isnull() & ~dt['ice_conc_temp_interp'].isnull()
if successfully_filled_nans_temp.any():
    print(f"Number of NaNs successfully filled by temporal interpolation: {successfully_filled_nans_temp.sum().item()}")
else:
    print("Temporal interpolation did not fill any NaNs. This indicates structural missingness (e.g., entire time slices or persistent region gaps).")


Temporal interpolation did not fill any NaNs. This indicates structural missingness (e.g., entire time slices or persistent region gaps).


In [ ]:
#spatial interpolation
